In [1]:
"""
Layer 2 odometry test.

Prerequisites:
  - Layer 1 notebook (01_test_connection) passed
  - Robot on the floor with open space around it (about 2m × 2m)
  - config.py has correct ARDUINO_IP

This notebook drives three known patterns and plots what odometry thinks
happened. If the plot matches reality, calibration is good.
"""

import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from robot_driver.robot    import Robot
from robot_driver.odometry import AckermannOdometry
from robot_driver          import config

print(f"Wheel radius:         {config.WHEEL_RADIUS_M} m")
print(f"Ticks per revolution: {config.ENCODER_TICKS_PER_REVOLUTION}")
print(f"Wheelbase:            {config.WHEELBASE_M} m")

Wheel radius:         0.035 m
Ticks per revolution: 40
Wheelbase:            0.2 m


In [2]:
def run_trajectory(robot, odom, commands, dt=0.05):
    """Execute a list of (speed, steering, duration_sec) commands,
    recording pose at each control step."""
    trajectory = [(odom.x, odom.y, odom.theta)]

    for speed, steering, duration in commands:
        t_end = time.time() + duration
        while time.time() < t_end:
            robot.drive(speed, steering)
            reading = robot.update_sensors()
            if reading is not None:
                x, y, th = odom.update(reading["encoder"], reading["steering"])
                trajectory.append((x, y, th))
            time.sleep(dt)

    robot.stop()
    time.sleep(0.2)
    return np.array(trajectory)

In [3]:
# Place robot in open space. Facing any direction, that's your new "forward".
robot = Robot()
odom  = AckermannOdometry()

# Drive forward slowly for ~2 seconds. Speed value is robot-specific; adjust.
commands = [
    (40, 0, 10.0),   # forward, straight, 2 seconds
]
traj = run_trajectory(robot, odom, commands)

plt.figure(figsize=(6, 6))
plt.plot(traj[:, 0], traj[:, 1], "b-", linewidth=2)
plt.plot(traj[0, 0], traj[0, 1], "go", markersize=10, label="start")
plt.plot(traj[-1, 0], traj[-1, 1], "ro", markersize=10, label="end")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title("Straight-line test")
plt.axis("equal"); plt.grid(True); plt.legend()
plt.show()

print(f"Odometry says: moved from ({traj[0,0]:.2f}, {traj[0,1]:.2f}) "
      f"to ({traj[-1,0]:.2f}, {traj[-1,1]:.2f})")
print(f"Distance traveled: {np.linalg.norm(traj[-1,:2] - traj[0,:2]):.2f} m")
print("Measure the actual distance with a tape — does it match?")

OSError: [Errno 65] No route to host

In [4]:
odom.reset()

# Drive forward with a constant right turn for several seconds.
commands = [
    (40, 20, 6.0),   # forward + steering right, for 6 seconds
]
traj = run_trajectory(robot, odom, commands)

plt.figure(figsize=(6, 6))
plt.plot(traj[:, 0], traj[:, 1], "b-", linewidth=2)
plt.plot(traj[0, 0], traj[0, 1], "go", markersize=10, label="start")
plt.plot(traj[-1, 0], traj[-1, 1], "ro", markersize=10, label="end")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title("Right turn test")
plt.axis("equal"); plt.grid(True); plt.legend()
plt.show()

OSError: [Errno 65] No route to host

In [ ]:
robot.close()
print("Done.")

In [ ]:
from robot_driver import config
print(f"ARDUINO_IP in config: {config.ARDUINO_IP}")
print(f"ARDUINO_PORT: {config.ARDUINO_PORT}")